# `DocLayNet` BiSeNet-ResNet18 Architecture
**Author**: Juan Pablo Triana Martinez.

The following notebook contains all of the `torch.nn` code to recreate a
**BiSeNet V1 (Bilateral Segmentation Network)** with a **ResNet18** backbone from
scratch (~**13.2M** parameters)!

- BiSeNet paper: https://arxiv.org/abs/1808.00897
- ResNet paper: https://arxiv.org/abs/1512.03385

BiSeNet runs **two parallel paths** and fuses them:
1. A **Spatial Path**: three stride-2 convolutions that keep rich spatial detail
   at 1/8 resolution with wide (128) channels.
2. A **Context Path**: the ResNet18 backbone with **Attention Refinement Modules
   (ARM)** on the 1/16 and 1/32 features, plus a global-average-pooling context tail.
3. A **Feature Fusion Module (FFM)** that concatenates both paths and applies a
   channel-attention residual, followed by the segmentation head and 8x upsampling.


## 1. The `ResNet18` encoder backbone (from scratch)

We first rebuild the **ResNet18** feature extractor from the original paper
(https://arxiv.org/abs/1512.03385), exactly as in `src/models/backbones.py`.
It is composed of:
- A **stem**: `7x7/2` convolution followed by `3x3/2` max pooling.
- Four residual stages (`layer1..layer4`), each with two `BasicBlock`s
  (two `3x3` convolutions + identity/projection skip connection).

For an input `(B, 3, 512, 512)` the encoder returns 5 multi-scale feature maps:

```python
x  -> stem_conv          -> f1 (B,  64, 256, 256)   # 1/2
f1 -> maxpool + layer1   -> f2 (B,  64, 128, 128)   # 1/4
f2 -> layer2             -> f3 (B, 128,  64,  64)   # 1/8
f3 -> layer3             -> f4 (B, 256,  32,  32)   # 1/16
f4 -> layer4             -> f5 (B, 512,  16,  16)   # 1/32
```

We start with a shared `ConvBNReLU` helper block used across all our benchmark architectures.


In [ ]:
# Let's import all necessary modules for this architecture
from typing import List
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
class ConvBNReLU(nn.Module):
    '''
    Standard Conv2d -> BatchNorm2d -> ReLU block used across all architectures.

    Args:
        m (int): number of input channels.
        n (int): number of output channels.
        kernel_size (int): convolution kernel size.
        stride (int): convolution stride.
        padding (int): convolution padding.
        dilation (int): convolution dilation.
        groups (int): convolution groups (used for depthwise convolutions).
        relu6 (bool): if True, uses ReLU6 (MobileNetV2 convention) instead of ReLU.
    '''

    def __init__(self, m: int, n: int, kernel_size: int = 3, stride: int = 1,
                 padding: int = 1, dilation: int = 1, groups: int = 1,
                 relu6: bool = False) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels=m, out_channels=n, kernel_size=kernel_size,
                      stride=stride, padding=padding, dilation=dilation,
                      groups=groups, bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True),
            nn.ReLU6() if relu6 else nn.ReLU()
        )

    def forward(self, x) -> torch.Tensor:
        return self.block(x)


In [ ]:
class ResNetBasicBlock(nn.Module):
    '''
    Class that defines the BasicBlock of the ResNet18 architecture
    (two 3x3 convolutions with an identity or projected skip connection).
    Reference: https://arxiv.org/abs/1512.03385

    Args:
        m (int): number of input channels.
        n (int): number of output channels.
        stride (int): stride of the first convolution (2 halves the resolution).
    '''

    def __init__(self, m: int, n: int, stride: int = 1) -> None:
        super().__init__()

        # First 3x3 convolution (possibly downsampling)
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=m, out_channels=n, kernel_size=(3, 3),
                      stride=(stride, stride), padding=(1, 1), bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True),
            nn.ReLU()
        )

        # Second 3x3 convolution (no activation before the residual add)
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=n, out_channels=n, kernel_size=(3, 3),
                      stride=(1, 1), padding=(1, 1), bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True)
        )

        # Projection skip connection when shape changes, identity otherwise
        if stride != 1 or m != n:
            self.skip_conn = nn.Sequential(
                nn.Conv2d(in_channels=m, out_channels=n, kernel_size=(1, 1),
                          stride=(stride, stride), padding=(0, 0), bias=False),
                nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                               affine=True, track_running_stats=True)
            )
        else:
            self.skip_conn = nn.Identity()

        self.relu = nn.ReLU()

    def forward(self, x) -> torch.Tensor:
        out = self.conv_block_1(x)
        out = self.conv_block_2(out)
        out = out + self.skip_conn(x)
        return self.relu(out)


In [ ]:
class ResNet18Encoder(nn.Module):
    '''
    Class that defines the full ResNet18 feature-extractor backbone from scratch
    (no fully connected head), returning multi-scale feature maps.
    Reference: https://arxiv.org/abs/1512.03385

    Feature maps returned for an input of shape (B, Cin, H, W):
        f1: (B,  64, H/2,  W/2)   -> after stem conv (before max pooling)
        f2: (B,  64, H/4,  W/4)   -> after layer1
        f3: (B, 128, H/8,  W/8)   -> after layer2
        f4: (B, 256, H/16, W/16)  -> after layer3
        f5: (B, 512, H/32, W/32)  -> after layer4

    Args:
        Cin (int): number of input channels (3 for RGB document images).
    '''

    # Output channels at each stage, useful for building decoders
    out_channels: List[int] = [64, 64, 128, 256, 512]

    def __init__(self, Cin: int = 3) -> None:
        super().__init__()

        # Stem: 7x7/2 convolution followed by 3x3/2 max pooling
        self.stem_conv = nn.Sequential(
            nn.Conv2d(in_channels=Cin, out_channels=64, kernel_size=(7, 7),
                      stride=(2, 2), padding=(3, 3), bias=False),
            nn.BatchNorm2d(num_features=64, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True),
            nn.ReLU()
        )
        self.max_pool = nn.MaxPool2d(kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))

        # Four residual stages, two BasicBlocks each (ResNet18 configuration)
        self.layer1 = nn.Sequential(
            ResNetBasicBlock(m=64, n=64, stride=1),
            ResNetBasicBlock(m=64, n=64, stride=1)
        )
        self.layer2 = nn.Sequential(
            ResNetBasicBlock(m=64, n=128, stride=2),
            ResNetBasicBlock(m=128, n=128, stride=1)
        )
        self.layer3 = nn.Sequential(
            ResNetBasicBlock(m=128, n=256, stride=2),
            ResNetBasicBlock(m=256, n=256, stride=1)
        )
        self.layer4 = nn.Sequential(
            ResNetBasicBlock(m=256, n=512, stride=2),
            ResNetBasicBlock(m=512, n=512, stride=1)
        )

    def forward(self, x) -> List[torch.Tensor]:
        f1 = self.stem_conv(x)          # (B, 64, H/2, W/2)
        f2 = self.layer1(self.max_pool(f1))  # (B, 64, H/4, W/4)
        f3 = self.layer2(f2)            # (B, 128, H/8, W/8)
        f4 = self.layer3(f3)            # (B, 256, H/16, W/16)
        f5 = self.layer4(f4)            # (B, 512, H/32, W/32)
        return [f1, f2, f3, f4, f5]


### 1.1 Summary info of `ResNet18Encoder`

In [ ]:
from torchinfo import summary
# Let's inspect the ResNet18 encoder backbone
test_model = ResNet18Encoder(Cin=3)

summary(model = test_model,
        input_size=(1, 3, 512, 512), # (batch_size, num_channels, height, width)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"],
        depth = 3
        )


## 2. The Spatial Path

Three stride-2 convolutions preserve spatial detail at high channel width:

```python
x (B, 3, 512, 512) -> 7x7/2 -> (B, 64, 256, 256)
                   -> 3x3/2 -> (B, 64, 128, 128)
                   -> 3x3/2 -> (B, 64,  64,  64)
                   -> 1x1   -> (B, 128, 64,  64)   # 1/8 resolution
```


In [ ]:
class SpatialPath(nn.Module):
    '''
    Class that defines the BiSeNet Spatial Path: three stride-2 convolutions
    that preserve rich spatial detail at 1/8 resolution with wide channels.

    Args:
        Cin (int): number of input channels.
        n (int): number of output channels (128 in the paper).
    '''

    def __init__(self, Cin: int = 3, n: int = 128) -> None:
        super().__init__()
        self.conv_7x7 = ConvBNReLU(m=Cin, n=64, kernel_size=7, stride=2, padding=3)
        self.conv_3x3_1 = ConvBNReLU(m=64, n=64, kernel_size=3, stride=2, padding=1)
        self.conv_3x3_2 = ConvBNReLU(m=64, n=64, kernel_size=3, stride=2, padding=1)
        self.conv_1x1 = ConvBNReLU(m=64, n=n, kernel_size=1, stride=1, padding=0)

    def forward(self, x) -> torch.Tensor:
        x = self.conv_7x7(x)      # 1/2 resolution
        x = self.conv_3x3_1(x)    # 1/4 resolution
        x = self.conv_3x3_2(x)    # 1/8 resolution
        return self.conv_1x1(x)   # (B, 128, H/8, W/8)


## 3. The Attention Refinement Module (ARM)

Global average pooling produces a per-channel attention vector (`1x1` conv +
BatchNorm + Sigmoid) that re-weights the feature map, letting the context path
emphasize the most informative channels at each scale.


In [ ]:
class AttentionRefinementModule(nn.Module):
    '''
    Class that defines the BiSeNet ARM: global average pooling produces a
    channel attention vector that re-weights the input feature map.

    Args:
        m (int): number of input channels.
        n (int): number of output channels.
    '''

    def __init__(self, m: int, n: int) -> None:
        super().__init__()
        self.conv = ConvBNReLU(m=m, n=n, kernel_size=3, stride=1, padding=1)
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(output_size=1),
            nn.Conv2d(in_channels=n, out_channels=n, kernel_size=(1, 1),
                      stride=(1, 1), padding=(0, 0), bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True),
            nn.Sigmoid()
        )

    def forward(self, x) -> torch.Tensor:
        x = self.conv(x)
        attention = self.attention(x)
        return x * attention


## 4. The Feature Fusion Module (FFM)

The spatial (128 ch) and context (128 ch) features are concatenated, projected
with a `1x1` conv, and refined with a channel-attention **residual**:
`out = x + x * attention(x)`.


In [ ]:
class FeatureFusionModule(nn.Module):
    '''
    Class that defines the BiSeNet FFM: concatenates the spatial and context
    features, projects them, and applies a channel attention residual.

    Args:
        m (int): number of input channels (spatial + context concatenated).
        n (int): number of output channels.
    '''

    def __init__(self, m: int, n: int) -> None:
        super().__init__()
        self.conv = ConvBNReLU(m=m, n=n, kernel_size=1, stride=1, padding=0)
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(output_size=1),
            nn.Conv2d(in_channels=n, out_channels=n // 4, kernel_size=(1, 1),
                      stride=(1, 1), padding=(0, 0), bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels=n // 4, out_channels=n, kernel_size=(1, 1),
                      stride=(1, 1), padding=(0, 0), bias=False),
            nn.Sigmoid()
        )

    def forward(self, x_spatial, x_context) -> torch.Tensor:
        x = torch.cat([x_spatial, x_context], dim=1)
        x = self.conv(x)
        attention = self.attention(x)
        return x + x * attention


## 5. Final step, let's create the entire network

The context path flows top-down:

```python
f5 (B, 512, 16, 16) -> ARM_32 + global context -> up 2x -> refine -> (B, 128, 32, 32)
f4 (B, 256, 32, 32) -> ARM_16 + previous       -> up 4x -> refine -> (B, 128, 64, 64)
FFM(spatial, context) -> (B, 256, 64, 64) -> head -> 8x upsample -> (B, N, 512, 512)
```


In [ ]:
class BiSeNetResNet18Model(nn.Module):
    '''
    Class that defines the full BiSeNet V1 architecture with a ResNet18
    context path. The Spatial Path keeps detail at 1/8 resolution while the
    Context Path (ResNet18 + global pooling + ARMs) provides a large
    receptive field; both are merged by the Feature Fusion Module.

    Args:
        Cin (int): number of input channels for both paths.
        N (int): number of output channels (1 binary / num_classes semantic).
    '''

    def __init__(self, Cin: int = 3, N: int = 1) -> None:
        super().__init__()

        # Spatial path: (B, 128, H/8, W/8)
        self.spatial_path = SpatialPath(Cin=Cin, n=128)

        # Context path: ResNet18 backbone with ARMs at 1/16 and 1/32
        self.context_path = ResNet18Encoder(Cin=Cin)
        _, _, _, c4, c5 = self.context_path.out_channels

        self.arm_16 = AttentionRefinementModule(m=c4, n=128)
        self.arm_32 = AttentionRefinementModule(m=c5, n=128)

        # Global context tail from the deepest feature
        self.global_context = nn.Sequential(
            nn.AdaptiveAvgPool2d(output_size=1),
            ConvBNReLU(m=c5, n=128, kernel_size=1, stride=1, padding=0)
        )

        # Refinement convolutions after each context upsampling
        self.refine_32 = ConvBNReLU(m=128, n=128, kernel_size=3, stride=1, padding=1)
        self.refine_16 = ConvBNReLU(m=128, n=128, kernel_size=3, stride=1, padding=1)

        # Feature fusion of spatial (128) + context (128) channels
        self.ffm = FeatureFusionModule(m=256, n=256)

        # Segmentation head at 1/8 resolution, then 8x upsampling
        self.head_conv = ConvBNReLU(m=256, n=256, kernel_size=3, stride=1, padding=1)
        self.segmentation_head = nn.Conv2d(in_channels=256, out_channels=N,
                                           kernel_size=(1, 1), stride=(1, 1),
                                           padding=(0, 0))

    def forward(self, x) -> torch.Tensor:
        # Spatial path detail features
        x_spatial = self.spatial_path(x)

        # Context path multi-scale features
        _, _, _, f4, f5 = self.context_path(x)

        # Global average pooling context, broadcast onto the 1/32 feature
        x_global = self.global_context(f5)

        # 1/32 branch: ARM + global context, upsample to 1/16 and refine
        x_32 = self.arm_32(f5) + x_global
        x_32 = F.interpolate(x_32, size=f4.shape[2:], mode="bilinear", align_corners=True)
        x_32 = self.refine_32(x_32)

        # 1/16 branch: ARM + upsampled 1/32 branch, upsample to 1/8 and refine
        x_16 = self.arm_16(f4) + x_32
        x_16 = F.interpolate(x_16, size=x_spatial.shape[2:], mode="bilinear", align_corners=True)
        x_context = self.refine_16(x_16)

        # Fuse both paths and predict at 1/8 resolution
        x_fused = self.ffm(x_spatial, x_context)
        logits = self.segmentation_head(self.head_conv(x_fused))

        # Upsample from 1/8 back to full input resolution
        return F.interpolate(logits, size=x.shape[2:], mode="bilinear", align_corners=True)


### 5.1 Summary with images of shape `(B * 3 * 1024 * 1024)`

In [ ]:
from torchinfo import summary
# Full BiSeNet-ResNet18 model at 1024x1024
test_model = BiSeNetResNet18Model(Cin=3, N=1)

summary(model = test_model,
        input_size=(1, 3, 1024, 1024), # (batch_size, num_channels, height, width)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"],
        depth = 3
        )


### 5.2 Summary with images of shape `(B * 3 * 512 * 512)`

In [ ]:
from torchinfo import summary
# Full BiSeNet-ResNet18 model at 512x512
test_model = BiSeNetResNet18Model(Cin=3, N=1)

summary(model = test_model,
        input_size=(1, 3, 512, 512), # (batch_size, num_channels, height, width)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"],
        depth = 3
        )


## Integration with `src/models` and the training framework

The exact same classes above live in `src/models`, and the `bisenet-resnet18` model can be
built through the shared model factory. This is what the training scripts use when you pass
`--arch bisenet-resnet18`:

```bash
python scripts/train_binary_text.py --arch bisenet-resnet18
python scripts/train_semantic_layout.py --arch bisenet-resnet18
```

Let's double check the factory produces the same model, and run the IEEE efficiency
benchmark (parameters, FLOPs, inference latency/FPS, and peak memory - GPU if available,
otherwise CPU RSS) with `src.utils.benchmark`.


In [ ]:
import sys
from pathlib import Path
# Allow imports from the project root (src.*)
sys.path.insert(0, str(Path().cwd().parent))

from src.models import build_model

factory_model = build_model("bisenet-resnet18", Cin=3, N=1)
total_params = sum(p.numel() for p in factory_model.parameters())
print(f"BiSeNet-ResNet18 total parameters: {total_params:,} ({total_params/1e6:.2f}M)")


In [ ]:
from src.utils import benchmark_model, print_benchmark

device = "cuda" if torch.cuda.is_available() else "cpu"
report = benchmark_model(
    model=factory_model,
    input_size=(1, 3, 512, 512),
    device=device,
    warmup=5,
    iterations=20,
    arch_name="bisenet-resnet18",
)
print_benchmark(report)
